In [156]:
pip install xmltodict lxml pydantic httpx


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import httpx
import json
import requests as r
import pandas as pd
from io import BytesIO
from zipfile import ZipFile
from data.flow import xml_survey
from typing import List, Optional
from util.util import readxml, Cipher
from util.flow import get_token, get_headers
from core.dev import Dev

In [ ]:
FLOW_SERVICE_URL="https://fiji-dws.akvoflow.org/"
flow_api = "https://api-auth0.akvo.org/flow/orgs"
flow_instances = pd.read_csv("./data/flow-survey-amazon-aws.csv")
USERNAME = os.environ["AUTH0_USER"]
PASSWORD = os.environ["AUTH0_PWD"]
INSTANCE_NAME = 'fiji-dws'

In [159]:
def get_folders(
    headers: dict,
    instance_name: str,
    id: Optional[int] = None,
):
    instance = flow_instances[flow_instances.instances.eq(instance_name)]
    if not instance.shape[0]:
        print("Instance Not Found")
        return
    url = f"{flow_api}/{instance_name}"
    folder_url = f"{url}/folders?parent_id=0"
    survey_url = f"{url}/surveys?folder_id=0"
    if id:
        folder_url = f"{url}/folders?parent_id={id}"
        survey_url = f"{url}/surveys?folder_id={id}"
    folders = r.get(folder_url, headers=headers)
    folders = folders.json().get("folders")
    surveys = r.get(survey_url, headers=headers)
    surveys = surveys.json().get("surveys")
    print({"folders": folders, "surveys": surveys})

In [160]:
def get_survey_forms(
    headers: dict,
    instance_name: str,
    id: int
):
    url = f"{flow_api}/{instance_name}/surveys/{id}"
    req = r.get(url, headers=headers)
    data = req.json().get("forms")
    # [d.update({"surveyId": id}) for d in data]
    return data

In [161]:
def download_sqlite_asset(cascade_list: List[str], ziploc: str) -> None:
    for cascade in cascade_list:
        cascade_file = cascade.split("/surveys/")[1]
        cascade_file = f"{ziploc}/{cascade_file}"
        cascade_file = cascade_file.replace(".zip", "")
        if not os.path.exists(cascade_file):
            try:
                zip_url = httpx.get(cascade)
                zip_url.raise_for_status()
                z = ZipFile(BytesIO(zip_url.content))
                z.extractall(ziploc)
            except httpx.HTTPError:
                pass
            # except httpx.HTTPError as exc:
            # raise HTTPException(
            #     status_code=exc.response.status_code,
            #     detail=f"Error while requesting {exc.request.url!r}.")

In [162]:
def download_form(ziploc: str, alias: str, survey_id: int):
    instance = xml_survey(alias)
    dev = Dev()
    xml_path = f"{ziploc}/{survey_id}.xml"
    if dev.get_cached(xml_path):
        return readxml(xml_path=xml_path, alias=alias)
    try:
        zip_url = httpx.get(f"{instance}/{survey_id}.zip")
        zip_url.raise_for_status()
    except httpx.HTTPError:
        return False
    if not os.path.exists(ziploc):
        os.mkdir(ziploc)
    z = ZipFile(BytesIO(zip_url.content))
    z.extractall(ziploc)
    response = readxml(xml_path=xml_path, alias=alias)
    cascade_list = []
    for qg in response["questionGroup"]:
        for q in qg.get("question", []):
            if q["type"] == "cascade":
                cascade_list.append(q["cascadeResource"])
    if len(cascade_list) > 0:
        cascade_list = [f"{instance}/{c}.zip" for c in cascade_list]
        download_sqlite_asset(cascade_list, ziploc)
    return response

In [163]:
def get_form(id: str, save_json: bool = True):
    alias, survey_id = Cipher(id).decode()
    if alias is None:
        print("Alias is None!")
        return
    ziploc = f"./static/xml/{alias}"
    form_data = download_form(ziploc, alias, survey_id)
    if not form_data:
        print("Form not found")
        return
    # Save as JSON if requested
    if save_json:
        output_dir = "./output"
        os.makedirs(output_dir, exist_ok=True)
        
        form_name = form_data.get("name", "unknown")
        # Sanitize the name for filename (replace invalid characters)
        safe_name = "".join(c if c.isalnum() or c in ('-', '_', ' ') else '_' for c in form_name)
        safe_name = safe_name.replace(' ', '_')
        
        output_file = f"{output_dir}/{survey_id}_{safe_name}.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(form_data, f, indent=2, ensure_ascii=False)
        print(f"Form saved to: {output_file}")
    return form_data

In [ ]:
def download_all_forms(form_ids: list = []):
    dev = Dev()
    for form_id in form_ids:
        form_chipper = dev.generate(
            alias=INSTANCE_NAME,
            fid=form_id
        )
        get_form(id=form_chipper)

In [164]:
res = get_token(username=USERNAME, password=PASSWORD)
refresh_token = res.get("refresh_token")
headers = get_headers(refresh_token)

In [165]:
form_ids = [
    8520967, # WTP
    17260923, # WWTP
    27040920, # SPS (Pump Station)
    1520924, # EPS Water quality
    5530933, # EPS Project Construction
    2490944, # RWS
]
download_all_forms(form_ids=form_ids)